# A tour of zarrista (for zarr-python users)

[**zarrista**](https://github.com/developmentseed/zarrista) is a small, low-level Zarr API for Python, powered from Rust by [**zarrs**](https://zarrs.dev/) via PyO3. It is inspired by [zarrita.js](https://zarrita.dev/): explicit, thin, and typed.

This notebook walks through the **current** zarrista API and maps each piece onto the equivalent **zarr-python** concept. Throughout, we use zarr-python and zarrista side by side on the **same data on disk** to show they are interoperable.

**Scope of this notebook:** the **synchronous** API on a **local filesystem** store only. zarrista also has a full `async` API (`AsyncArray` / `AsyncGroup`) and object-store / Icechunk backends, which are not covered here.

> zarrista is an **evaluation prototype**, not production-ready. Some APIs (notably writing) are intentionally minimal.

## Setup

We import NumPy, zarr-python (for comparison / fixtures), and the zarrista names we'll use. Everything runs in a throwaway temporary directory.

In [3]:
import tempfile
from pathlib import Path

import numpy as np
import zarr  # zarr-python, for comparison and to write fixtures

from zarrista import (
    Array,
    ArrayBuilder,
    ArrayBytes,
    ChunkGrid,
    DataType,
    FillValue,
    FilesystemStore,
    Group,
    MemoryStore,
    codec,
)

tmp = Path(tempfile.mkdtemp())
print("zarr-python:", zarr.__version__)
print("working dir:", tmp)

zarr-python: 3.2.1
working dir: /var/folders/42/5jr6891d4ds4xysz7q0rsghw0000gn/T/tmpgqk4gxa8


## 1. Opening an array written by zarr-python

First we write a Zarr v3 array with **zarr-python**, exactly as a user normally would. zarrista will open the *same bytes on disk*.

In [4]:
path = tmp / "temperature.zarr"
data = np.arange(9 * 64 * 100, dtype="int32").reshape(9, 64, 100)

z = zarr.create_array(
    store=str(path),
    shape=data.shape,
    chunks=(3, 16, 50),
    dtype=data.dtype,
)
z[:] = data
z

<Array file:///var/folders/42/5jr6891d4ds4xysz7q0rsghw0000gn/T/tmpgqk4gxa8/temperature.zarr shape=(9, 64, 100) dtype=int32>

Now open it with zarrista. Where zarr-python uses `zarr.open_array(store)`, zarrista uses `Array.open(store)`, and the store is an explicit `FilesystemStore`.

In [5]:
arr = Array.open(FilesystemStore(path))

print("shape:           ", arr.shape)
print("ndim:            ", arr.ndim)
print("dtype:           ", arr.dtype)
print("dimension_names: ", arr.dimension_names)
print("path:            ", arr.path)

shape:            [9, 64, 100]
ndim:             3
dtype:            DataType(int32 / <i4)
dimension_names:  None
path:             /


### Attribute mapping

| concept | zarr-python | zarrista |
|---|---|---|
| open array | `zarr.open_array(store)` | `Array.open(store)` |
| store | `LocalStore(path)` / a path | `FilesystemStore(path)` |
| shape | `z.shape` (tuple) | `arr.shape` (list) |
| dtype | `z.dtype` (numpy) | `arr.dtype` (`DataType`) |
| dims | `z.metadata.dimension_names` | `arr.dimension_names` |
| attrs | `z.attrs` | `arr.attrs` |
| raw metadata | `z.metadata` | `arr.metadata` (v3 JSON dict) |

## 2. Reading data

zarrista supports **NumPy-style basic indexing** via `arr[...]` (or the explicit `arr.retrieve_array_subset(selection)`). Indexing returns a **decoded-array result object**, not a NumPy array directly — call `.to_numpy()` to get the array.

Supported today: integers, step-1 slices, negative indices, and `...`. Not yet: `step != 1`, `None`/newaxis, boolean and fancy indexing.

In [ ]:
result = arr[0:2, :, 5:7]  # zarrista: returns a Tensor (a DecodedArray)
print("result type:", type(result).__name__)
print("result.shape:", result.shape, "result.dtype:", result.dtype)

region = result.to_numpy()  # zero-copy NumPy view over Rust memory
expected = data[0:2, :, 5:7]  # the equivalent plain-numpy slice
print("matches numpy slice:", np.array_equal(region, expected))

result type: Tensor
result.shape: [2, 64, 2] result.dtype: DataType(int32 / <i4)
matches numpy slice: True


`arr[...]` is exactly equivalent to `arr.retrieve_array_subset(...)`; the indexing operator just forwards to it.

In [11]:
selection = (slice(0, 2), slice(None), slice(5, 7))
by_method = arr.retrieve_array_subset(selection).to_numpy()
print("arr[...] == retrieve_array_subset(...):", np.array_equal(region, by_method))

# Whole array, and an int index (note: zarrs-style, an int keeps a length-1 axis)
print("arr[...].shape (full):", arr[...].shape)
print("arr[5].shape (int keeps axis):", arr[5].shape)

arr[...] == retrieve_array_subset(...): True
arr[...].shape (full): [9, 64, 100]
arr[5].shape (int keeps axis): [1, 64, 100]


### Why a result object instead of a bare NumPy array?

A read returns one of four `DecodedArray` types depending on the data's memory layout: `Tensor` (fixed-width, dense), `VariableArray` (strings/bytes), `MaskedTensor`, `MaskedVariableArray`. The decoded bytes live in Rust memory and are exposed **zero-copy** through multiple protocols:

- **buffer protocol** → `result.buffer()` / `np.frombuffer(...)`
- **NumPy** → `result.to_numpy()`
- **DLPack** → `np.from_dlpack(result)` (carries shape natively)

There is no copy on the Rust→Python boundary; the decode allocation *is* the array's memory.

In [ ]:
# Raw buffer (1-D bytes) -> reinterpret with the known dtype and shape
flat = np.frombuffer(result.buffer(), dtype="int32").reshape(result.shape)
print("from buffer matches:", np.array_equal(flat, expected))

# DLPack carries the shape, so it comes back N-D directly
via_dlpack = np.from_dlpack(result)
print(
    "from_dlpack shape:",
    via_dlpack.shape,
    "matches:",
    np.array_equal(via_dlpack, expected),
)

from buffer matches: True
from_dlpack shape: (2, 64, 2) matches: True


### Reading individual chunks

Because zarrista is low-level, you can address the chunk grid directly — there is no exact zarr-python public equivalent for this. `retrieve_chunk` decodes a chunk; `retrieve_encoded_chunk` hands back the raw stored (pre-codec) bytes.

In [ ]:
chunk = arr.retrieve_chunk([0, 0, 0])  # the chunk at grid index (0, 0, 0)
print("chunk shape:", chunk.shape)
print(
    "matches data[0:3, 0:16, 0:50]:",
    np.array_equal(chunk.to_numpy(), data[0:3, 0:16, 0:50]),
)

encoded = arr.retrieve_encoded_chunk([0, 0, 0])  # raw stored bytes
print("encoded chunk: %d bytes" % len(bytes(encoded)))

chunk shape: [3, 16, 50]
matches data[0:3, 0:16, 0:50]: True
encoded chunk: 5320 bytes


## 3. Inspecting metadata and codecs

zarrista exposes the codec pipeline directly, split into the three Zarr v3 roles: **filters** (array→array), the **serializer** (array→bytes), and **compressors** (bytes→bytes).

In [8]:
print("chunk_grid.grid_shape:", arr.chunk_grid.grid_shape)
print("chunk_grid.array_shape:", arr.chunk_grid.array_shape)
print("filters:    ", [c.name for c in arr.filters])
print("serializer: ", arr.serializer.name)
print("compressors:", [c.name for c in arr.compressors])
print("\nfull v3 metadata:")
arr.metadata

chunk_grid.grid_shape: [3, 4, 2]
chunk_grid.array_shape: [9, 64, 100]
filters:     []
serializer:  bytes
compressors: ['zstd']

full v3 metadata:


{'zarr_format': 3,
 'node_type': 'array',
 'shape': [9, 64, 100],
 'data_type': 'int32',
 'chunk_grid': {'name': 'regular',
  'configuration': {'chunk_shape': [3, 16, 50]}},
 'chunk_key_encoding': {'name': 'default',
  'configuration': {'separator': '/'}},
 'fill_value': 0,
 'codecs': [{'name': 'bytes', 'configuration': {'endian': 'little'}},
  {'name': 'zstd', 'configuration': {'level': 0, 'checksum': False}}]}

## 4. Creating an array with `ArrayBuilder`

Where zarr-python has `zarr.create_array(...)`, zarrista uses an **immutable, chained builder**. Every setter returns a *new* builder, so configuration is explicit and reusable. The required pieces — chunk grid, data type, fill value — are constructor arguments; everything else is an optional setter.

In [ ]:
out = tmp / "created.zarr"

new = (
    ArrayBuilder(
        ChunkGrid.regular([8, 8], [4, 4]),  # array shape [8,8], chunks [4,4]
        DataType.from_string("int32"),
        FillValue(np.int32(0).tobytes()),  # fill value as native-endian bytes
    )
    .dimension_names(["y", "x"])
    .compressors([codec.zstd(3, checksum=False)])
    .attrs({"units": "kelvin"})
    .create(FilesystemStore(out), "/")  # materialize + write metadata
)

print("created:", new.shape, new.dtype, new.dimension_names)
print("compressors:", [c.name for c in new.compressors])

created: [8, 8] DataType(int32 / <i4) ['y', 'x']
compressors: ['zstd']


Writing data today is at the **chunk** level: `store_chunk` takes the chunk's grid index and an `ArrayBytes` wrapping the raw decoded bytes. (Multi-chunk region writes — the equivalent of `z[a:b] = ...` — are not yet implemented.)

In [ ]:
block = np.full((4, 4), 7, dtype="int32")
new.store_chunk([0, 0], ArrayBytes(block.tobytes()))

# Read it back with zarrista...
roundtrip = Array.open(FilesystemStore(out)).retrieve_chunk([0, 0]).to_numpy()
print("zarrista read-back matches:", np.array_equal(roundtrip, block))

# ...and confirm zarr-python reads what zarrista wrote (full interoperability)
zp = zarr.open_array(str(out))
print(
    "zarr-python reads zarrista's chunk:",
    np.array_equal(np.asarray(zp[0:4, 0:4]), block),
)

zarrista read-back matches: True
zarr-python reads zarrista's chunk: True


### Sharding

Sharding is modelled as the array→bytes serializer, selected by setting an inner (subchunk) shape on the builder — there is no separate `shards=` keyword. We can inspect the resulting metadata without touching a store via `create_metadata()`.

In [ ]:
sharded_meta = (
    ArrayBuilder(
        ChunkGrid.regular([8, 8], [4, 4]),
        DataType.from_string("int32"),
        FillValue(np.int32(0).tobytes()),
    )
    .subchunk_shape([2, 2])  # inner shards of 2x2 -> sharding serializer
    .create_metadata()
)
print("serializer is:", sharded_meta["codecs"][0]["name"])

serializer is: sharding_indexed


## 5. Groups

We build a small group hierarchy with **zarr-python**, then navigate it with zarrista's `Group`.

In [12]:
gpath = tmp / "dataset.zarr"
root = zarr.open_group(str(gpath), mode="w")
root.create_array("temperature", shape=(4, 4), chunks=(2, 2), dtype="float32")
nested = root.create_group("diagnostics")
nested.create_array("pressure", shape=(2,), chunks=(2,), dtype="float64")

g = Group.open(FilesystemStore(gpath))
print("array_keys:", g.array_keys())
print("group_keys:", g.group_keys())
print("traverse:  ", [type(n).__name__ for n in g.traverse()])

# Index into a group like zarr-python: g[name] -> Array or Group
child = g["temperature"]
print("g['temperature'] ->", type(child).__name__, child.shape)

array_keys: ['temperature']
group_keys: ['diagnostics']
traverse:   ['Array', 'Group', 'Array']
g['temperature'] -> Array [4, 4]


| concept | zarr-python | zarrista |
|---|---|---|
| open group | `zarr.open_group(store)` | `Group.open(store)` |
| child arrays | `list(g.array_keys())` | `g.array_keys()` |
| child groups | `list(g.group_keys())` | `g.group_keys()` |
| access child | `g[name]` | `g[name]` / `g.child(name)` |
| recurse | `g.members(...)` | `g.traverse()` |

Group reading is complete; group **creation** is not yet implemented in zarrista.

## 6. The in-memory store

`MemoryStore` is a drop-in store for tests and scratch work — no filesystem involved.

In [ ]:
mem = ArrayBuilder(
    ChunkGrid.regular([2, 2], [2, 2]),
    DataType.from_string("uint8"),
    FillValue(b"\x00"),
).create(MemoryStore(), "/scratch")
print("in-memory array:", mem.shape, mem.dtype)

in-memory array: [2, 2] DataType(uint8 / |u1)


## Summary

What this notebook exercised, all interoperable with zarr-python on the same files:

- **Read**: `Array.open`, `arr[...]` / `retrieve_array_subset`, `retrieve_chunk`, `retrieve_encoded_chunk`, with zero-copy NumPy / buffer / DLPack exchange.
- **Inspect**: `shape`, `dtype`, `dimension_names`, `chunk_grid`, `filters` / `serializer` / `compressors`, full v3 `metadata`.
- **Create + write**: `ArrayBuilder` → `create`, `store_chunk` (chunk-level), sharding via `subchunk_shape`.
- **Groups**: `Group.open`, `array_keys` / `group_keys` / `traverse`, `g[name]`.
- **Stores**: `FilesystemStore`, `MemoryStore`.

**Not covered here** (but present or planned): the full `async` API (`AsyncArray` / `AsyncGroup`), object-store and Icechunk backends, variable-length / masked result types' `to_numpy()`, multi-chunk region writes, and group creation.